In [0]:
import requests
from pyspark.sql import Row

class DatabricksPipelineManager:
    def __init__(self, base_url, token):
        self.base_url = base_url
        self.token = token

    def list_pipelines(self):
        """
        Lists all Lakeflow Declarative Pipelines (DLT pipelines) in the workspace.
        Returns a list of tuples containing pipeline_id and pipeline name.
        """
        Pipeline = Row("pipeline_id", "pipeline_name")

        url = f"{self.base_url}/api/2.0/pipelines/"
        headers = {"Authorization": f"Bearer {self.token}"}
        print(url, headers)
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            print("Pipeline successfully.")
            return [Pipeline(p['pipeline_id'], p['name']) for p in response.json()['statuses']]
        else:
            print(f"Error {response.status_code}: {response.text}")


    def get_pipeline_id(self, pipeline_name):
        """
        Retrieves the pipeline_id for the given pipeline_name.
        Returns None if not found.
        """
        pipelines = self.list_pipelines()
        for pipeline in pipelines:
            if pipeline['pipeline_name'] == pipeline_name:
                return pipeline['pipeline_id']
        return None

    def delete_pipeline(self, pipeline_id):
        """
        Deletes the Lakeflow Declarative Pipeline with the specified pipeline_id.
        Cascade option deletes dependent resources (set to false to retain tables).
        """
        url = f"{self.base_url}/api/2.0/pipelines/{pipeline_id}"
        headers = {"Authorization": "Bearer {token}"}
        params = {"cascade": "true"}  # Set to false to keep tables

        response = requests.delete(url, headers=headers, params=params)

        if response.status_code == 200:
            print("Pipeline deleted successfully.")
        else:
            print(f"Error {response.status_code}: {response.text}")

In [0]:
### Test
base_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
pipeline_mgr = DatabricksPipelineManager(base_url, token)
pipeline_mgr.list_pipelines()
pipeline_mgr.get_pipeline_id('employees-etl')